# 10 - Orquestador del Pipeline (ejecución desde un notebook)

Trabajo Práctico - Pipeline ETL end-to-end (Bronze -> Silver / DQL -> Gold)

El pipeline tiene **dos formas de orquestación**, que respetan las mismas dependencias:

1. **Job de Databricks** (`job_pipeline_datos_empresas.json`). Es la orquestación principal: cada notebook es una tarea, el Job muestra el grafo de dependencias (la imagen del pipeline que se entrega) y se puede programar.
2. **Este notebook**: ejecuta el mismo flujo con `dbutils.notebook.run`. Lo usa `11_Prueba_Idempotencia` para correr el pipeline varias veces seguidas, y sirve para ejecutar todo a mano sin crear el Job.

## Grafo de dependencias

```
01_Configuracion_Pipeline
        |
        +--> 02_Silver_CIIU --> 03_Silver_RUES --> 06_Gold_Dim_Empresas --+
        +--> 04_Silver_SECOP ---------------------------------------------+--> 07_Gold_Fact_Contratos --> 08_Gold_Agg_Indicadores_Mes --> 09_Auditoria_Ejecucion
        `--> 05_Silver_TRM -----------------------------------------------+
```

Las tareas de una misma etapa no dependen entre sí, así que se ejecutan **en paralelo**. Una etapa empieza solo cuando terminó bien la anterior. Si una tarea falla, el pipeline se detiene y no se construye Gold con datos incompletos.

In [ ]:
%python
import json
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime

dbutils.widgets.text("id_ejecucion", "", "Id de ejecución (vacío = automático)")
ID_EJECUCION = dbutils.widgets.get("id_ejecucion").strip() or f"orquestador_{datetime.now():%Y%m%d_%H%M%S}"

TIMEOUT_SEGUNDOS = 3600

ETAPAS = [
    ["01_Configuracion_Pipeline"],
    ["02_Silver_CIIU", "04_Silver_SECOP", "05_Silver_TRM"],
    ["03_Silver_RUES"],
    ["06_Gold_Dim_Empresas"],
    ["07_Gold_Fact_Contratos"],
    ["08_Gold_Agg_Indicadores_Mes"],
    ["09_Auditoria_Ejecucion"],
]

print(f"id_ejecucion = {ID_EJECUCION}")

In [ ]:
%python
def ejecutar(notebook: str) -> dict:
    inicio = datetime.now()
    salida = dbutils.notebook.run(f"./{notebook}", TIMEOUT_SEGUNDOS, {"id_ejecucion": ID_EJECUCION})
    segundos = (datetime.now() - inicio).total_seconds()
    print(f"  OK {notebook} ({segundos:.0f} s)")
    return {"notebook": notebook, "segundos": round(segundos), "salida": json.loads(salida) if salida else None}


resultados = []
inicio_total = datetime.now()

for numero, etapa in enumerate(ETAPAS, start=1):
    print(f"Etapa {numero}/{len(ETAPAS)}: {', '.join(etapa)}")
    with ThreadPoolExecutor(max_workers=len(etapa)) as pool:
        # .result() relanza la excepción de cualquier tarea fallida y detiene el pipeline
        resultados.extend(f.result() for f in [pool.submit(ejecutar, nb) for nb in etapa])

print(f"\nPipeline completo en {(datetime.now() - inicio_total).total_seconds():.0f} s")

In [ ]:
%python
display(spark.createDataFrame([(r["notebook"], r["segundos"]) for r in resultados], "notebook string, segundos long"))

dbutils.notebook.exit(json.dumps({"id_ejecucion": ID_EJECUCION, "estado": "OK"}))